# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.
     

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.
     

3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.
     

4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.
     

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.
     

6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.

### Part 1 - My answers

**E1 · Data vs. competence**
No. 100 successes in a row only show that the robot works in the situation that was filmed (same mug, same fill level, same room). The video does not tell us what happens if something changes, for example a different kitchen or a fuller mug. So it does not prove the robot can do the task in any home. My confidence that it proves this: **1 / 5**.

**E2 · Changed case**
If the mug is nearly full, the coffee has almost no space to move before it reaches the rim. During a turn the coffee sloshes and **spills over the edge**. The untested assumption in the first run: the fill level (and so the slosh) stays like in the half-full test, so the same speed and turn are still safe.

**E3 · Commitment decision**
The robot should **ask** Eva. The evidence: it is not confirmed that Eva accepts the white mug. This is only in Eva's head, so the robot cannot find it out by looking (observe), and proceeding would be a guess. Asking is quick and cheap. (In the warrant gate this is the *grounded* check that fails.)

**E4 · Five questions (T1-T5)**

| | Question |
|---|---|
| T1 Explain | Why did the robot choose the white mug and a slow speed? Which beliefs and which limit caused this? |
| T2 Predict | If the mug is nearly full, what slosh height does the robot predict in the turn, and will it spill? *(changed condition)* |
| T3 Design | Which speed, tilt and route should the robot choose if the room is crowded and Eva has moved? |
| T4 Bound | For which fill levels, speeds and mugs does the successful run still hold? Where must the robot not commit? *(changed condition)* |
| T5 Check | What does the robot measure while carrying (coffee level, force, Eva's position) to see that its prediction was wrong, and when does it stop? |

T2 and T4 ask about changed conditions that a video alone cannot answer.

**E5 · System decomposition**

| Module | What it does for Eva's request | Box in the integration diagram | Code (Part 2) |
|---|---|---|---|
| Interpreting | turns "bring me my coffee" into a concrete goal | interpret & ground | `InterpretationSystem` |
| Perceiving | finds out where Eva is, which mug, temperature | perceive (reading → belief) | `SensingSystem`, `PerceptionSystem`, `BeliefState` |
| Proposing | suggests speed, tilt, grasp | propose (System 1 · learning) | `Proposer` |
| Predicting | predicts force and slosh for the proposal | know & predict (digital twin, System 2) | `SemanticDigitalTwin` |
| Warranting | decides if the action may be executed | decide: the warrant gate (System 2) | `WarrantGate` |
| Acting | moves the body, carries the mug | act & regulate | `ExecutionEngine.execute_action` |
| Recording | saves what was asked, believed, committed, observed | record & justify | `ExecutionEngine.record_trace` |

**E6 · Eva stands up**
- *Revised action:* pause, keep the mug level, look at Eva again and plan a new handover position (higher, where she stands now). Move slowly.
- *Possible failure:* the old target position is wrong and the sudden change makes the coffee slosh and spill (or the robot gets too close to Eva).
- *Evidence needed before committing:* Eva's pose is stable (for example speed below 0.1 m/s for about 1 s), and the digital twin predicts that slosh stays under the limit on the new path with at least 0.5 m distance to Eva. If not, wait and check again. (The numbers are examples.)

## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

### Part 2 - Code

I used the given code and made these changes:

| Where | Change | Task |
|---|---|---|
| `Proposer` | completed with Gaussian sampling | 1.2 |
| `SemanticDigitalTwin.predict_properties` | dual-stage stiffness, viscosity depends on temperature, battery drain | 2.1, 3.1, 4.1 |
| `WarrantGate.validate` | 7th check `energetically_viable` | 4.1 |
| `BeliefState` | `battery_level_pct`, `battery_reserve_pct` | 4.2 |
| `run_delivery` | one full decision cycle, recharge if battery too low | 1.1, 4.2 |
| `audit_analyzer` | analysis of the audit trail | 5.1 |

I also found two problems in the given demo and fixed them:
1. The demo printed "REJECTED" / "APPROVED" by hand. With its numbers, attempt 2 would really fail the gate (wave about 3.3 mm, limit 1.5 mm), but it still printed "APPROVED". Now the real result is printed.
2. `predicted_nav_velocity = min(v, limit)` can never be above the limit, so the speed check never failed. Now it uses the requested speed `v`.

In [1]:
from dataclasses import dataclass
from typing import Dict, Any, Tuple, Optional, List
import random
import time

# ---------- data classes ----------
@dataclass(frozen=True)
class GoalSpecification:
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float
    trajectory_length_m: float = 8.0      # new: needed for the battery estimate (Task 4)

@dataclass
class ParameterPacket:
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "stochastic_sampler"

@dataclass
class Observation:
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str

@dataclass
class BeliefState:
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"
    preferred_cup: str = "blue_ceramic_mug"
    battery_level_pct: float = 100.0      # new (Task 4.2)
    battery_reserve_pct: float = 10.0     # new (Task 4.2)

# ---------- Task 1.2: Proposer (System 1) ----------
class Proposer:
    """Samples velocity and tilt rate from Gaussian distributions."""
    def __init__(self, v_mean=0.3, v_std=0.15, tilt_mean=0.06, tilt_std=0.04):
        self.start = (v_mean, tilt_mean)
        self.v_mean, self.v_std = v_mean, v_std
        self.tilt_mean, self.tilt_std = tilt_mean, tilt_std

    def propose(self) -> ParameterPacket:
        v = random.gauss(self.v_mean, self.v_std)
        tilt = random.gauss(self.tilt_mean, self.tilt_std)
        v = min(max(v, 0.05), 1.0)         # keep the values physically sensible
        tilt = min(max(tilt, 0.01), 0.3)
        return ParameterPacket((0.45, 0.12, 0.30), round(v, 3), round(tilt, 3), "gaussian_sampler")

    def make_safer(self):
        """Called after a rejection: move the distribution to slower values."""
        self.v_mean = self.v_mean * 0.7
        self.tilt_mean = self.tilt_mean * 0.7

    def reset(self):
        self.v_mean, self.tilt_mean = self.start

# ---------- interpretation, sensing, perception (unchanged from the assignment) ----------
class InterpretationSystem:
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        print(f"[INTERPRETATION] '{raw_utterance}'")
        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})
        busyness_factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)
        max_nav_vel = 1.2 * busyness_factor
        is_urgent = "schnell" in raw_utterance.lower() or "fast" in raw_utterance.lower()
        time_frame = 30.0 if is_urgent else 60.0
        target_volume = 15.0 if not is_urgent else 12.0
        goal = GoalSpecification(cup_type, belief.eva_position, target_volume,
                                 cup_properties["max_force"], cup_properties["max_wave"],
                                 max_nav_vel, time_frame)
        print(f" -> Goal: {goal.target_object}, MaxForce={goal.max_allowable_force_n} N, "
              f"MaxWave={goal.max_slosh_wave_mm} mm, MaxNavVel={goal.max_nav_velocity_mps} m/s")
        return goal

class SensingSystem:
    def sense(self, true_ambient_temp, true_friction, eva_pos, busyness) -> Observation:
        return Observation(time.time(),
                           round(true_ambient_temp + random.uniform(-0.5, 0.5), 2),
                           round(true_friction + random.uniform(-0.05, 0.05), 2),
                           eva_pos, busyness)

class PerceptionSystem:
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness

# ---------- Semantic Digital Twin (System 2) ----------
class SemanticDigitalTwin:
    def __init__(self, belief_state: BeliefState, noise: bool = True):
        self.belief = belief_state
        self.noise = noise          # noise=False -> no random error, good for tests

    def contact_depth(self, v):
        # assumption: depth grows with approach speed (0.08 m per m/s)
        return 0.08 * v

    def predict_properties(self, params: ParameterPacket, goal: GoalSpecification) -> Dict[str, float]:
        v = params.max_velocity_mps

        # Task 2.1: dual-stage stiffness
        k = 12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)
        depth = self.contact_depth(v)
        if depth > 0.02:                              # bottoming out
            k = k * 1.8 ** (depth / 0.02)             # grows exponentially
        predicted_force = k * (v ** 1.5)

        # Task 3.1: viscosity drops 1% per degC above 20 degC
        effective_viscosity = self.belief.fluid_viscosity_pas * (1.0 - (self.belief.ambient_temperature_c - 20.0) * 0.01)
        radius = 0.05
        predicted_wave = (params.tilt_rate_radps * radius) / effective_viscosity * 1000.0

        # Task 4.1: battery drain in % of total capacity
        duration_s = goal.trajectory_length_m / v
        power_w = 150.0 + 40.0 * v ** 2
        drain_pct = power_w * duration_s / 3600.0 / 100.0 * 100.0     # capacity = 100 Wh

        force_error = random.uniform(-0.2, 0.3) if self.noise else 0.0
        wave_error = random.uniform(-0.1, 0.2) if self.noise else 0.0
        return {
            "predicted_force_n": round(max(0.0, predicted_force + force_error), 2),
            "predicted_wave_mm": round(max(0.0, predicted_wave + wave_error), 2),
            "predicted_nav_velocity": v,   # fixed: the old min(v, limit) could never fail
            "predicted_battery_drain_pct": round(drain_pct, 3),
        }

# ---------- Warrant Gate (Task 4.1: 7th check) ----------
class WarrantGate:
    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    def validate(self, params, goal, belief) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal)
        drain = predictions["predicted_battery_drain_pct"]
        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (predictions["predicted_force_n"] <= goal.max_allowable_force_n) and
                    (predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm) and
                    (predictions["predicted_nav_velocity"] <= goal.max_nav_velocity_mps),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled,
            # new: drain below 15% AND enough battery left (reserve 10%)
            "energetically_viable": drain < 15.0 and (belief.battery_level_pct - drain) >= belief.battery_reserve_pct,
        }
        failed = [name for name, ok in checks.items() if not ok]
        return {"passed": len(failed) == 0,
                "failed_criterion": failed[0] if failed else None,
                "failed_criteria": failed,
                "predictions": predictions,
                "warrant_checks": checks}

# ---------- Execution + audit trail ----------
class ExecutionEngine:
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params, goal, warrant_result) -> Dict[str, Any]:
        print(f"[ACTUATOR] executing v={params.max_velocity_mps} m/s, tilt={params.tilt_rate_radps} rad/s")
        time.sleep(0.01)
        pred = warrant_result["predictions"]
        actual_force = pred["predicted_force_n"] + random.uniform(-0.3, 0.6)
        actual_wave = pred["predicted_wave_mm"] + random.uniform(-0.2, 0.5)
        spill = (actual_force > goal.max_allowable_force_n) or (actual_wave > goal.max_slosh_wave_mm)
        return {"actual_force_n": round(actual_force, 2), "actual_wave_mm": round(actual_wave, 2),
                "actual_volume_transferred_ml": goal.target_volume_ml - (3.0 if spill else 0.0),
                "spill_occurred": spill, "status": "FAILURE" if spill else "SUCCESS"}

    def record_trace(self, asked_utterance, goal, belief, proposal, warrant_result, observation,
                     committed, execution_outcome=None, action_type="deliver_coffee"):
        self.audit_trail.append({
            "timestamp": time.time(),
            "asked": {"raw_utterance": asked_utterance,
                      "interpreter_output": {"target": goal.target_object,
                                             "max_force_limit_n": goal.max_allowable_force_n,
                                             "max_nav_velocity": goal.max_nav_velocity_mps}},
            "belief": {"eva_position": belief.eva_position, "room_busyness": belief.room_busyness,
                       "ambient_temp_c": belief.ambient_temperature_c,
                       "surface_friction": belief.surface_friction_coefficient,
                       "battery_level_pct": belief.battery_level_pct},
            "observed": {"timestamp": observation.timestamp,
                         "measured_temp_c": observation.measured_temperature_c,
                         "measured_friction": observation.measured_friction},
            "committed": {"action_approved": committed, "action_type": action_type,
                          "proposed_parameters": {"max_velocity_mps": proposal.max_velocity_mps,
                                                  "tilt_rate_radps": proposal.tilt_rate_radps},
                          "warrant_verdict": warrant_result},
            "execution_outcome": execution_outcome})

# ---------- One full decision cycle (Task 1.1 / 4.2) ----------
def run_delivery(command, belief, proposer, interpreter, gate, executive,
                 true_temp=21.5, busyness="medium", max_tries=15):
    sensing, perception = SensingSystem(), PerceptionSystem(belief)

    # 1. sense + perceive
    obs = sensing.sense(true_temp, 0.45, "living_room", busyness)
    perception.update(obs)
    # 2. interpret
    goal = interpreter.interpret(command, belief)
    # 3-5. propose -> predict -> warrant, repeat until approved
    proposer.reset()
    for attempt in range(1, max_tries + 1):
        proposal = proposer.propose()
        warrant = gate.validate(proposal, goal, belief)
        p = warrant["predictions"]
        print(f"attempt {attempt}: v={proposal.max_velocity_mps}, tilt={proposal.tilt_rate_radps} | "
              f"F={p['predicted_force_n']} N, wave={p['predicted_wave_mm']} mm, drain={p['predicted_battery_drain_pct']} % "
              f"-> {'APPROVED' if warrant['passed'] else 'REJECTED ' + str(warrant['failed_criteria'])}")
        if warrant["passed"]:
            # 6. act + record
            outcome = executive.execute_action(proposal, goal, warrant)
            executive.record_trace(command, goal, belief, proposal, warrant, obs, True, outcome)
            return "delivered"
        executive.record_trace(command, goal, belief, proposal, warrant, obs, False)
        if "energetically_viable" in warrant["failed_criteria"]:      # Task 4.2
            print("[REDIRECT] battery too low -> asking for a recharge instead of delivering coffee")
            outcome = {"status": "RECHARGE_REQUESTED", "spill_occurred": False}
            executive.record_trace(command, goal, belief, proposal, warrant, obs, True, outcome, "recharge_request")
            return "recharge_requested"
        proposer.make_safer()          # rejected because of "safe" -> try slower
    return "no_plan_found"

# ---------- Task 5.1 ----------
def audit_analyzer(audit_trail):
    deliveries = [e for e in audit_trail if e["committed"]["action_type"] == "deliver_coffee"]
    rejected = [e for e in deliveries if not e["committed"]["action_approved"]]
    approved = [e for e in deliveries if e["committed"]["action_approved"]]

    # 2) count which criterion failed most often
    counts = {}
    for e in rejected:
        for name, ok in e["committed"]["warrant_verdict"]["warrant_checks"].items():
            if not ok:
                counts[name] = counts.get(name, 0) + 1
    most_common = max(counts, key=counts.get) if counts else None

    # 3) predicted vs observed force
    errors = []
    for e in approved:
        predicted = e["committed"]["warrant_verdict"]["predictions"]["predicted_force_n"]
        observed = e["execution_outcome"]["actual_force_n"]
        errors.append(observed - predicted)
    mean_err = sum(errors) / len(errors) if errors else 0.0
    variance = sum((x - mean_err) ** 2 for x in errors) / len(errors) if errors else 0.0

    print("1) rejected proposals:", len(rejected), "| approved executions:", len(approved))
    print("2) rejections per criterion:", counts, "-> most frequent:", most_common)
    print(f"3) force error (observed - predicted): mean = {mean_err:.3f} N, variance = {variance:.3f} N^2")
    return counts, mean_err, variance

## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition
* **Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.
* **Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.
  

### Task 2: Non-Linear Contact Mechanics and Material Compliance
In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation. 
* **Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds $0.02 \, \text{m}$, the stiffness coefficient should increase exponentially by a factor of $1.8$ due to bottoming out against a rigid internal fixture.
* **Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity ($0.35 \, \text{m/s}$). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.

### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling
Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.
* **Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient ($1\% \, \text{drop per} \, ^\circ\text{C}$ above $20^\circ\text{C}$).
* **Question 3.2:** Simulate an environment where the ambient temperature rises to $28^\circ\text{C}$. How does this change the predicted wave height for a tilt rate of $0.15 \, \text{rad/s}$? Does the Warrant Gate trigger a safety violation?

### Task 4: Expanding the Warrant Gate Checklist
The current Warrant Gate evaluates six core criteria (*grounded, feasible, safe, monitorable, repairable, recordable*). 
* **Question 4.1:** Add a seventh criterion called **`energetically_viable`** to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below $15\%$ of total capacity.
* **Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

### Task 5: Post-Hoc Epistemic Audit Analysis
A core pillar of trustworthy robotics is the ability to audit *why* a decision was made or rejected.
* **Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:
  1. The total number of rejected proposals versus approved executions.
  2. The most frequent warrant criterion responsible for rejection.
  3. The average variance between predicted forces and observed physical outcomes.
* **Question 5.2:** Discuss in your lab report: *Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?*

### Task 1

**Question 1.1 - Mapping and data flow**

| Box in the diagram | Class / method |
|---|---|
| interpret & ground | `InterpretationSystem.interpret` |
| perceive | `SensingSystem.sense`, `PerceptionSystem.update`, `BeliefState` |
| propose (System 1) | `Proposer.propose` |
| know & predict (System 2) | `SemanticDigitalTwin.predict_properties` |
| decide: warrant gate (System 2) | `WarrantGate.validate` |
| act & regulate | `ExecutionEngine.execute_action` |
| record & justify | `ExecutionEngine.record_trace` |

Data flow in one cycle (this is what `run_delivery` does):
1. The sensors give a noisy `Observation`. `PerceptionSystem` uses it to update the `BeliefState`.
2. The `InterpretationSystem` reads the command and the belief and makes a `GoalSpecification` (limits for force, slosh, speed).
3. The `Proposer` suggests a `ParameterPacket` (speed, tilt).
4. The `SemanticDigitalTwin` predicts force, wave height and battery drain for this packet.
5. The `WarrantGate` compares the predictions with the goal and the belief (7 checks). If one fails, the packet is rejected and the Proposer tries again (or the robot asks for a recharge).
6. If all checks pass, the `ExecutionEngine` executes the action and `record_trace` saves asked, belief, committed and observed.

**Question 1.2 - Proposer**
I used a **Gaussian** distribution for the speed and the tilt rate (values are cut to a sensible range). After a rejection the means are made smaller, so the next proposal is slower and gentler.

Why Gaussian: a uniform distribution would also give many extreme values (like 1 m/s with a full mug) and has no idea what is a normal value. A Gaussian puts most samples around a normal value but still tries other values sometimes. A mixture of Gaussians would be useful with two clear modes (careful and fast), but here I only have one normal operating point. The Proposer does not need to be perfect because the Warrant Gate checks every proposal anyway.

In [2]:
random.seed(1)
KB = {"cups": {"blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
               "plastic_cup":      {"max_force": 7.0, "max_wave": 3.0}},
      "room_busyness_limits": {"low": 1.0, "medium": 0.6, "high": 0.3}}

# the proposer starts with fast values (like the original demo: 0.8 m/s) so we can see the rejections
belief = BeliefState()
executive = ExecutionEngine()
result = run_delivery("Bring me my coffee fast!", belief,
                      Proposer(v_mean=0.8, v_std=0.1, tilt_mean=0.2, tilt_std=0.05),
                      InterpretationSystem(KB), WarrantGate(SemanticDigitalTwin(belief)), executive)
print("RESULT:", result)

[INTERPRETATION] 'Bring me my coffee fast!'
 -> Goal: blue_ceramic_mug, MaxForce=4.5 N, MaxWave=1.5 mm, MaxNavVel=0.72 m/s
attempt 1: v=0.807, tilt=0.162 | F=63.38 N, wave=6.86 mm, drain=0.485 % -> REJECTED ['safe']
attempt 2: v=0.458, tilt=0.068 | F=11.77 N, wave=2.77 mm, drain=0.769 % -> REJECTED ['safe']
attempt 3: v=0.447, tilt=0.052 | F=11.38 N, wave=2.09 mm, drain=0.785 % -> REJECTED ['safe']
attempt 4: v=0.124, tilt=0.095 | F=0.49 N, wave=4.19 mm, drain=2.699 % -> REJECTED ['safe']
attempt 5: v=0.212, tilt=0.041 | F=1.09 N, wave=1.79 mm, drain=1.591 % -> REJECTED ['safe']
attempt 6: v=0.225, tilt=0.015 | F=1.31 N, wave=0.66 mm, drain=1.501 % -> APPROVED
[ACTUATOR] executing v=0.225 m/s, tilt=0.015 rad/s
RESULT: delivered


### Task 2

**Question 2.1 - Dual-stage compliance**
In `predict_properties`: the contact depth is `0.08 * v` (my own assumption, because the original code has no depth). Up to a depth of 0.02 m the stiffness is the normal `k`. Above 0.02 m the pouch hits the rigid part inside, so I use `k = k * 1.8 ** (depth / 0.02)`. This means the stiffness is 1.8 times higher at the threshold and grows exponentially when the pouch is pressed deeper. The force is still `F = k * v^1.5`.
(If only a fixed factor 1.8 is meant, the result is 4.87 N, which is also above the limit.)

**Question 2.2 - Test with 0.35 m/s**

In [3]:
belief = BeliefState()
sdt = SemanticDigitalTwin(belief, noise=False)      # no random error, so the test is repeatable
gate = WarrantGate(sdt)
goal = GoalSpecification("blue_ceramic_mug", "living_room", 15.0, 4.5, 1.5, 0.72, 60.0)

packet = ParameterPacket((0.45, 0.12, 0.30), max_velocity_mps=0.35, tilt_rate_radps=0.03)  # small tilt: only force can fail
old_force = 12.5 * (1 + 0.1 * belief.surface_friction_coefficient) * 0.35 ** 1.5
verdict = gate.validate(packet, goal, belief)

print("force with the old single-stage model :", round(old_force, 2), "N (would pass the 4.5 N limit)")
print("force with the dual-stage model       :", verdict["predictions"]["predicted_force_n"], "N")
print("failed checks                         :", verdict["failed_criteria"])
print("approved?                             :", verdict["passed"])

assert not verdict["passed"] and verdict["failed_criterion"] == "safe"
print("TEST PASSED: the gate rejected the packet before anything was executed")

force with the old single-stage model : 2.7 N (would pass the 4.5 N limit)
force with the dual-stage model       : 6.16 N
failed checks                         : ['safe']
approved?                             : False
TEST PASSED: the gate rejected the packet before anything was executed


**Result:** the predicted force is about **6.2 N**, the limit is 4.5 N, so the Warrant Gate rejects the packet (`safe` fails) before the robot moves. With the old model the force would only be 2.7 N and the packet would be wrongly approved.

### Task 3

**Question 3.1 - Temperature and viscosity**
In `predict_properties` the viscosity gets 1 % lower for each °C above 20 °C: `eta = 1.2 * (1 - (T - 20) * 0.01)`. (I used the linear option from the task. The given code already had this formula; I checked that it works.) The wave height is `tilt * radius / eta`, so a lower viscosity gives a higher wave.

**Question 3.2 - 28 °C and tilt rate 0.15 rad/s**

In [4]:
goal = GoalSpecification("blue_ceramic_mug", "living_room", 15.0, 4.5, 1.5, 0.72, 60.0)
packet = ParameterPacket((0.45, 0.12, 0.30), max_velocity_mps=0.2, tilt_rate_radps=0.15)  # v=0.2 keeps the force small

for temp in (20.0, 28.0):
    belief = BeliefState(ambient_temperature_c=temp)
    gate = WarrantGate(SemanticDigitalTwin(belief, noise=False))
    result = gate.validate(packet, goal, belief)
    eta = 1.2 * (1 - (temp - 20) * 0.01)
    max_tilt = 1.5 * eta / 0.05 / 1000        # largest tilt rate that still keeps the wave below 1.5 mm
    print(f"T={temp} C: viscosity={eta:.3f}, wave={result['predictions']['predicted_wave_mm']} mm, "
          f"safe={result['warrant_checks']['safe']}, max safe tilt rate={max_tilt:.4f} rad/s")

T=20.0 C: viscosity=1.200, wave=6.25 mm, safe=False, max safe tilt rate=0.0360 rad/s
T=28.0 C: viscosity=1.104, wave=6.79 mm, safe=False, max safe tilt rate=0.0331 rad/s


**Answer:** at 28 °C the viscosity is 1.10 Pa·s (at 20 °C it is 1.20), so the wave is higher: about **6.8 mm instead of 6.25 mm** (+8.6 %). **Yes, the Warrant Gate rejects it** (`safe` fails) because the limit for the blue mug is 1.5 mm. Note: 0.15 rad/s is also too high at 20 °C. The temperature makes the safe tilt rate smaller (0.036 → 0.033 rad/s), so a setting that is only just safe in a cool room can become unsafe in a warm room.

### Task 4

**Question 4.1 - `energetically_viable`**
Battery drain model: time = distance / speed, power = 150 W + 40 W·v², capacity = 100 Wh, so drain % = energy / capacity. The check passes if (1) the drain is below 15 % and (2) the battery after the trip is still at least the reserve of 10 %. I added the second part because otherwise a robot with 8 % battery would pass (its trip only needs about 1.4 %) and the low-battery case of Question 4.2 would not fail.

In [5]:
goal = GoalSpecification("blue_ceramic_mug", "living_room", 15.0, 4.5, 1.5, 0.72, 60.0)
for battery, v in [(100, 0.25), (8, 0.25), (100, 0.02)]:
    belief = BeliefState(battery_level_pct=battery)
    gate = WarrantGate(SemanticDigitalTwin(belief, noise=False))
    r = gate.validate(ParameterPacket((0.45, 0.12, 0.30), v, 0.03), goal, belief)
    print(f"battery={battery}%, v={v} m/s -> drain={r['predictions']['predicted_battery_drain_pct']} %, "
          f"energetically_viable={r['warrant_checks']['energetically_viable']}")
print("(1) normal  (2) low battery  (3) very slow robot, trip takes so long that drain > 15 %")

battery=100%, v=0.25 m/s -> drain=1.356 %, energetically_viable=True
battery=8%, v=0.25 m/s -> drain=1.356 %, energetically_viable=False
battery=100%, v=0.02 m/s -> drain=16.668 %, energetically_viable=False
(1) normal  (2) low battery  (3) very slow robot, trip takes so long that drain > 15 %


**Question 4.2 - Low battery**
I added `battery_level_pct` to the `BeliefState`. In `run_delivery`, if `energetically_viable` fails, the robot does not try new speeds (a new speed cannot create battery). It stores the rejected coffee proposal and then a `recharge_request` action in the audit trail.

In [6]:
random.seed(5)
belief = BeliefState(battery_level_pct=8.0)
executive = ExecutionEngine()
result = run_delivery("Bring me my coffee!", belief, Proposer(), InterpretationSystem(KB),
                      WarrantGate(SemanticDigitalTwin(belief)), executive)
print("RESULT:", result)
print("audit trail:", [(e["committed"]["action_type"], e["committed"]["action_approved"]) for e in executive.audit_trail])

[INTERPRETATION] 'Bring me my coffee!'
 -> Goal: blue_ceramic_mug, MaxForce=4.5 N, MaxWave=1.5 mm, MaxNavVel=0.72 m/s
attempt 1: v=0.4, tilt=0.01 | F=8.65 N, wave=0.6 mm, drain=0.869 % -> REJECTED ['safe', 'energetically_viable']
[REDIRECT] battery too low -> asking for a recharge instead of delivering coffee
RESULT: recharge_requested
audit trail: [('deliver_coffee', False), ('recharge_request', True)]


### Task 5

**Question 5.1 - `audit_analyzer`**
It goes through the audit trail and prints (1) how many proposals were rejected and how many were approved and executed, (2) how often each criterion caused a rejection and which one is the most frequent, (3) the difference between observed and predicted force (mean and variance). I run several episodes first so that the trail has enough entries.

In [7]:
random.seed(7)
KB_belief = BeliefState()
executive = ExecutionEngine()
proposer = Proposer(v_mean=0.6, v_std=0.2, tilt_mean=0.12, tilt_std=0.06)    # starts a bit too fast
interpreter = InterpretationSystem(KB)
gate = WarrantGate(SemanticDigitalTwin(KB_belief))

for command, temp, battery in [("Bring me my coffee fast!", 21.5, 100), ("Bring me my coffee!", 21.5, 100),
                               ("Bring me my coffee fast!", 28.0, 100), ("Bring me my coffee!", 24.0, 100),
                               ("Bring me my coffee!", 21.5, 8), ("Bring me my coffee!", 20.0, 100)]:
    KB_belief.battery_level_pct = battery
    print("\n---", command, "| temp", temp, "| battery", battery)
    run_delivery(command, KB_belief, proposer, interpreter, gate, executive, true_temp=temp)

print("\n=== AUDIT ANALYSIS ===")
audit_analyzer(executive.audit_trail)


--- Bring me my coffee fast! | temp 21.5 | battery 100
[INTERPRETATION] 'Bring me my coffee fast!'
 -> Goal: blue_ceramic_mug, MaxForce=4.5 N, MaxWave=1.5 mm, MaxNavVel=0.72 m/s
attempt 1: v=0.555, tilt=0.101 | F=19.93 N, wave=4.27 mm, drain=0.65 % -> REJECTED ['safe']
attempt 2: v=0.642, tilt=0.109 | F=30.13 N, wave=4.63 mm, drain=0.576 % -> REJECTED ['safe']
attempt 3: v=0.373, tilt=0.07 | F=7.14 N, wave=3.1 mm, drain=0.927 % -> REJECTED ['safe']
attempt 4: v=0.307, tilt=0.071 | F=4.67 N, wave=3.18 mm, drain=1.113 % -> REJECTED ['safe']
attempt 5: v=0.05, tilt=0.01 | F=0.43 N, wave=0.34 mm, drain=6.671 % -> APPROVED
[ACTUATOR] executing v=0.05 m/s, tilt=0.01 rad/s

--- Bring me my coffee! | temp 21.5 | battery 100
[INTERPRETATION] 'Bring me my coffee!'
 -> Goal: blue_ceramic_mug, MaxForce=4.5 N, MaxWave=1.5 mm, MaxNavVel=0.72 m/s
attempt 1: v=0.468, tilt=0.223 | F=12.41 N, wave=9.47 mm, drain=0.754 % -> REJECTED ['safe']
attempt 2: v=0.296, tilt=0.04 | F=4.28 N, wave=1.6 mm, drain=1

({'safe': 22, 'energetically_viable': 1},
 0.21200000000000002,
 0.04785600000000001)

Most rejections are `safe` because the proposer starts with values that are too fast. The one `energetically_viable` rejection is the low-battery episode. The force error is small: in this simulation the real force is the predicted force plus a random value, so the digital twin is slightly optimistic. The variance shows how big the gap between simulation and reality is.

**Question 5.2 - Why keep `asked`, `belief`, `committed` and `observed` separate?**
Imitation learning only stores (observation → action). It cannot tell us *what* was asked, *what* the robot believed, *why* it was allowed to act, or *what really happened*. With four separate records we can find out where an error came from:
- `asked`: did the robot understand the request correctly?
- `belief`: was its picture of the world right (temperature, friction, Eva's position)?
- `committed`: why did it act (or refuse)? The predictions and the 7 checks are the reason. Rejected attempts are saved too.
- `observed`: what really happened, so we can compare it with the prediction (simulation-to-reality gap).

So we can say if a failure came from a wrong interpretation, a wrong belief, a wrong prediction or the execution, and fix the right part. A pure imitation policy cannot give this explanation.

## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

---

### Part 3 - Robot anatomy

In the picture the **blue lines** point to parts of the robot's body (structure and actuation) and the **orange lines** point to sensors. The robot is a mobile manipulator: a wheeled base, a torso, a head with a camera, and arms with grippers.

| Part in the picture | Category | What it does | How it supports the architecture |
|---|---|---|---|
| **Arm** (links joined by joints) | links + joints | Links are the rigid segments, joints connect them and give the degrees of freedom. The arm moves the hand to the mug. | The proposed grasp pose and tilt rate are carried out by moving the joints. Joint limits matter for the `feasible` check. The digital twin uses the links and joints to predict the motion. |
| **Gripper** (hand) | effector | Touches and holds the mug. | Its force is the contact force that the twin predicts and the gate checks (`safe`). |
| **Electric motor** | actuator | Turns a command into torque and speed in a joint or wheel. | `ExecutionEngine.execute_action` is realized by the motors. Their limits restrict speed and tilt rate. |
| **Wheel** | locomotion (wheels instead of legs) | Moves the base through the room. Wheels are fast and efficient on flat floors; legs would be better for stairs. | Gives the navigation speed (limited by `max_nav_velocity`) and uses battery (`energetically_viable`). |
| **Camera on the head** (RGB-D, the picture shows what it sees) | sensor, exteroception | Sees the objects, the coffee level and Eva. | Gives the observation for perception and the belief. It is needed for the `monitorable` check. |
| **Laser scanner** on the base (shows a distance of 1.42 m to the desk) | sensor, exteroception | Measures the distance to obstacles. | Used for navigation and collision avoidance, and updates the belief about the room. |
| **Torso, head, base** (visible on the robot) | body parts | The base carries everything and moves it, the torso carries the arms and head, the head carries the camera. | The base gives mobility, the torso can change the height (for example the handover height), the head gives the point of view for perception. |
| **Cable** (thin red line) | wiring | Power and data connection between parts. | Connects the sensors and motors to the computer where the architecture runs. |

Short comparisons asked in the title:
- **Links vs joints:** links are the rigid bodies, joints are the moving connections.
- **Actuators vs sensors:** actuators act (output), sensors measure (input). Together they close the perception-action loop.
- **Effectors vs base / torso / head:** the effector (gripper) touches and changes the world; base, torso and head position the effector and the sensors.
- **Wheels vs legs vs grippers:** wheels and legs are for moving the whole robot, a gripper is for handling objects.
- **Proprioception vs exteroception vs interoception:** proprioception senses the robot's own body (joint angles, motor current), exteroception senses the world (camera, laser scanner), interoception senses the internal state (battery level, motor temperature). The camera and the laser scanner in the picture are exteroceptive. The battery level in my belief state is interoceptive.

*(Note to myself: in the picture there is also a white box connected to the base, I could not clearly see what it is. Check the picture and add it to the table if needed.)*

# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 

*Note on AI use: I used Claude (an AI assistant) to help understand the tasks and to draft parts of this notebook, as allowed in the course.*